# 1. Pipeline walkthrough

A guided tour of LifeJacket's decision logic. **No API key or GCP access is
needed for most of this notebook** -- the deterministic parts (severity,
duplicates, dispatch) are plain Python, and the context APIs are free and
keyless.

Read [`backend/lifejacket/services/pipeline.py`](../backend/lifejacket/services/pipeline.py)
alongside this notebook; it is the same flow in production code.

## The nine steps

| # | Step | Where |
|---|---|---|
| 1 | Photo + location submitted | `api/routes/intake.py` |
| 2 | Tide, weather, place resolved | `context/environment.py` |
| 3 | Animal identified -- agent's own questions, max 5 | `agents/identification.py`, `taxonomy.py` |
| 4 | Condition assessed (<=5 questions) | `agents/assessment.py` |
| 5 | **Severity computed deterministically** | `dispatch/severity.py` |
| 6 | Report written | `agents/report.py` |
| 7 | Report checked for safety + grounding | `agents/guardrail.py` |
| 8 | Duplicates linked | `dispatch/duplicates.py` |
| 9 | Responders ranked for coordinator approval | `dispatch/matching.py` |

To **talk to** the chatbots, see notebooks 3 (identification), 4 (assessment), and 5 (report and guardrail).

In [ ]:
# Auto-apply backend code edits without restarting the kernel: after you
# (or Claude) edit a .py file under backend/lifejacket/, the next cell you
# run picks up the change automatically -- no "Restart Kernel" needed.
#
# This does NOT cover prompts/*.md (those are cached by load_prompt(), not
# reloaded as Python modules) -- call clear_prompt_cache() for those, or use
# the cell further down that already does this after a prompt edit.
%load_ext autoreload
%autoreload 2

# Make the backend package importable from the notebooks/ directory.
import sys, pathlib

REPO_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'backend'))

print('repo root:', REPO_ROOT)

## Step 2: environmental context

This is the part the earlier prototype was missing entirely. Tide state is
usually the single most decisive fact about a stranded marine mammal:

- **Rising** tide may refloat the animal with no intervention.
- **Falling** tide is a hard deadline -- heat stress starts, and the animal
  is left further from the water every minute.

Try changing the coordinates to your own coastline.

In [ ]:
from lifejacket.context.environment import gather_context, summarise_for_prompt
from lifejacket.models.schemas import GeoPoint

# Moss Landing, CA -- a real stranding hotspot on Monterey Bay.
location = GeoPoint(latitude=36.8044, longitude=-121.7869)
context = gather_context(location)

# This exact text is what the assessment agent reads.
print(summarise_for_prompt(context))

In [ ]:
# Anything that failed is named, so the report can say 'tide unknown'
# rather than implying the tide was checked and found unremarkable.
print('coastal:', context.is_coastal)
print('unavailable:', context.unavailable or 'nothing -- all lookups succeeded')

## Step 6: severity scoring

**Severity is deterministic, not an LLM call.** The model decides what it
*observes*; this function decides what those observations *mean*.

That split exists so a coordinator can be shown exactly which flags fired
and what each contributed. "The model said critical" is not reviewable.

All weights live in [`config/scoring.json`](../config/scoring.json).

In [ ]:
from lifejacket.dispatch.severity import score_severity, recommended_action
from lifejacket.models.schemas import (
    AnimalGroup, InjuryAssessment, MobilityConcern, SituationHazards,
)

# An entangled sea lion.
injury = InjuryAssessment(injury_present=True, entanglement=True, wound=True)
result = score_severity(
    injury=injury,
    hazards=SituationHazards(near_people=True),
    animal_group=AnimalGroup.PINNIPED,
    context=context,
)

print('level      :', result.level.value)
print('score      :', result.score)
print('confidence :', result.confidence)
print('hard rules :', result.triggered_rules)
print()
for flag, points in result.contributions.items():
    print(f'  {flag:<22} +{points}')
print()
for reason in result.reasons:
    print(' -', reason)

### Why entanglement needs a hard rule

Additively, entanglement scores only 1.4 -- below the 3.0 "respond"
threshold. An entangled animal with no other visible symptoms would be
triaged as "monitor" while the line tightens around it.

So some conditions bypass the score entirely. Compare:

In [ ]:
for label, inj, group in [
    ('entangled only', InjuryAssessment(entanglement=True), AnimalGroup.PINNIPED),
    ('immobile cetacean', InjuryAssessment(mobility_concern=MobilityConcern.IMMOBILE), AnimalGroup.CETACEAN),
    ('immobile pinniped', InjuryAssessment(mobility_concern=MobilityConcern.IMMOBILE), AnimalGroup.PINNIPED),
    ('minor swelling', InjuryAssessment(swelling=True), AnimalGroup.PINNIPED),
]:
    r = score_severity(injury=inj, hazards=SituationHazards(), animal_group=group)
    rules = ', '.join(r.triggered_rules) or '-'
    print(f'{label:<20} {r.level.value:<10} score={r.score:<5} rules={rules}')

Note the third row. A seal resting on a beach is **normal behaviour**; a
dolphin that cannot move is suffocating under its own weight. Same flag,
different animal, opposite urgency -- which is why `animal_group` is an
argument to the scorer.

## Step 10: responder matching

Scored on area coverage (weight 0.58) and capability (0.42). Distance is
used to order and to estimate an ETA, but is **not** part of the match
score -- a nearby organisation with no permit for the species is not a
better match than a qualified one an hour away.

In [ ]:
from lifejacket.context.geocode import reverse_geocode
from lifejacket.dispatch.matching import rank_centers

place = reverse_geocode(36.8044, -121.7869)
print('county      :', place.county)
print('area terms  :', place.area_terms)
print()

for candidate in rank_centers(
    area_terms=place.area_terms,
    animal_group=AnimalGroup.PINNIPED,
    injury=injury,
    latitude=36.8044,
    longitude=-121.7869,
):
    print(f'{candidate.score:.2f}  {candidate.name}')
    print(f'      {candidate.rationale}')
    print()

### Exercise

The state name is deliberately excluded from `area_terms`. Add `'california'`
to the list above and re-run: a centre 400 km away in Del Norte County
starts scoring as highly as the local one, because every entry in the
California directory contains the word "California".

This was a real bug found while building the pipeline. County and locality
are the granularity at which coverage is actually assigned.

## Step 9: duplicate detection

One stranded animal on a public beach gets reported by everyone who walks
past. Without this, a single sea lion becomes fifteen incidents.

In [ ]:
from datetime import datetime, timedelta
from lifejacket.dispatch.duplicates import CandidateIncident, duplicate_confidence, find_duplicate

print('Confidence by distance and time (same species):')
for metres, hours in [(50, 0.2), (50, 6), (400, 1), (900, 20), (2000, 1)]:
    c = duplicate_confidence(metres / 1000, hours, True)
    print(f'  {metres:>5} m, {hours:>4} h -> {c:.3f}')

In [ ]:
now = datetime.now()
match = find_duplicate(
    latitude=36.8044, longitude=-121.7869,
    reported_at=now,
    animal_group=AnimalGroup.PINNIPED,
    candidates=[
        CandidateIncident('INC_earlier', 36.8046, -121.7871, now - timedelta(minutes=25), AnimalGroup.PINNIPED),
        CandidateIncident('INC_faraway', 37.5000, -122.2000, now, AnimalGroup.PINNIPED),
    ],
)
print(match)

## Steps 4, 5, 7, 8: the agents

These need Vertex AI credentials:

```bash
gcloud auth application-default login
```

The cells below are the real call paths. Each agent loads its prompt from
[`prompts/`](../prompts/) and returns validated JSON.

In [ ]:
# Prompts are Markdown files, not Python strings, so a prompt change shows
# up as a readable diff in code review.
from lifejacket.llm.prompts import list_prompts, load_prompt

for path in list_prompts():
    print(path.relative_to(REPO_ROOT / 'prompts'))

In [ ]:
print(load_prompt('system_principles.md')[:1500])

In [ ]:
# Uncomment once credentials are set up.
#
# from lifejacket.agents.identification import IdentificationAgent
# from lifejacket.llm.client import ImageInput
#
# photo = ImageInput(data=pathlib.Path('your_photo.jpg').read_bytes())
# agent = IdentificationAgent()
# result = agent.run(
#     images=[photo],
#     location_summary=summarise_for_prompt(context),
#     questions_remaining=5,
# )
#
# for candidate in result.candidates:
#     print(f'{candidate.confidence:.2f}  {candidate.common_name}')
# print('confident:', result.is_confident)
# print('next question:', result.next_question)

## Running the whole thing

```bash
uvicorn lifejacket.api.main:app --reload --app-dir backend
```

Then open <http://localhost:8000/docs> and drive the intake flow by hand:
`POST /intake/start`, upload a photo, post a location, and answer the
questions as they come back.